In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q scipy
!pip install -q h5py
!pip install -q matplotlib
!pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install -q transformers
!pip install -q fuzzy_match
!pip install -q nltk
!pip install -q rouge
!pip install -q diffusers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 760.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 60.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchdata 0.7.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
torchtext 0.16.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.5 MB/s eta 0:00:00


In [3]:
%cd /content/drive/MyDrive/Research/FINAL/Code/MMMM

/content/drive/MyDrive/Research/FINAL/Code/MMMM


In [4]:
import sys
sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")
sys.path.append("./trainer")

import torch
import torch.nn as nn
import torch.optim as optim

from master_init import *
from DSG import *

In [19]:
config = {
    "device" : "cuda:0",
    "device_ids" : [0]
}
device = config["device"]
device_ids = config["device_ids"]

model = INITIALIZE_MODEL(device=device, device_ids=device_ids).to(device)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


In [6]:
# dataset_dict = INITIALIZE_DATALOADERS(
#     keys=["ZuCo-BART", "Brain2Image"],
#     bsz=[64, 1]
# )

In [10]:
dataset_dict = INITIALIZE_DATALOADERS(
    keys=["ZuCo-BART"],
    bsz=[64]
)

In [7]:
dsg_tasks = DSGTasks()
dsg_tasks.add_task(
    DSGTask(
        task_name="EEG-TXT-BART-SENTIMENT",
        dataset_tag="ZuCo-BART",
        criterion=nn.CrossEntropyLoss(), # CE Loss for Tenary Sentiment
        optimizer=optim.Adam,
        learning_rate=5e-3,
        converge_lim=2,
        converge_threshold=0.005,
        div_threshold=0.01
        )
    )

## Code

In [9]:
# # dataloader = args_dict["dataloader"]
# # model = args_dict["model"]
# # optimizer = args_dict["optimizer"]
# # tokenizer = args_dict["tokenizer"]
# # criterion = args_dict["criterion"]
# # device = args_dict["device"] if "device" in args_dict else "cuda"
# # device_ids = args_dict["device_ids"] if "device_ids" in args_dict else None
# # staging_device = args_dict["staging_device"] if "staging_device" in args_dict else None

# dataloader = dataset_dict["ZuCo-BART"]
# model = model
# optimizer = optim.Adam(model.parameters(), lr=5e-3)
# from transformers import BartTokenizer
# tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
# criterion = nn.CrossEntropyLoss()
# device = "cuda"
# device_ids = None

In [10]:
# staging_device = "cuda:0"
# if staging_device==None:
#     staging_device = f"cuda:{device_ids[0]}" if device_ids == None else "cuda"
# results = {}
# for phase in ['train', 'dev']:
#     if phase == 'train':
#         model.train()    # Set model to training mode
#     else:
#         model.eval()     # Set model to evaluate mode

#     running_loss = 0.0
#     tot_cnt = 0

#     # Iterate over data.
#     current_data = dataloader[phase].load_data()
#     while not current_data["reset"]:
#         input_embeddings, seq_len, input_masks, input_mask_invert, target_ids, target_mask, sentiment_labels, sent_level_EEG = current_data["data"]

#         good = []
#         for i in range(len(sentiment_labels)):
#             if not sentiment_labels[i] == -100:
#                 good.append(i)

#         if len(good) == 0:
#             current_data = dataloader[phase].load_data()
#             continue

#         input_embeddings = input_embeddings[good]
#         input_masks = input_masks[good]
#         input_mask_invert = input_mask_invert[good]
#         target_ids = target_ids[good]
#         sentiment_labels = sentiment_labels[good]

#         target = torch.zeros(input_embeddings.shape[0], 3).to(device)
#         for i in range(input_embeddings.shape[0]):
#             target[i][sentiment_labels[i]] = 1.0

#         input_embeddings_batch = input_embeddings.to(staging_device).float()
#         input_masks_batch = input_masks.to(staging_device)
#         input_mask_invert_batch = input_mask_invert.to(staging_device)
#         target_ids_batch = target_ids.to(staging_device)

#         """replace padding ids in target_ids with -100"""
#         target_ids_batch[target_ids_batch == tokenizer.pad_token_id] = -100

#         optimizer.zero_grad()

#         args_dict = {
#             "input_data_batch" : input_embeddings_batch,
#             "input_masks_batch" : input_masks_batch,
#             "input_masks_invert" : input_mask_invert_batch,
#             "target_ids_batch" : target_ids_batch,
#             "pool_result" : True
#             }

#         output = model(
#             mode="EEG-TEXT-BART-SENTIMENT",
#             args_dict=args_dict,
#             staging_device=staging_device
#             )

#         loss = criterion(output.to(dtype=float), target.to(dtype=float))

#         # Backward + Optimize only if in training phase
#         if phase == 'train':
#             if device_ids == None:
#                     loss.backward()
#                     optimizer.step()
#             else:
#                     loss.mean().backward()
#                     optimizer.step()

#         # Compute stats
#         if device_ids == None:
#             running_loss += loss.item() * input_embeddings_batch.size()[0]
#         else:
#             running_loss += loss.mean().item() * input_embeddings_batch.size()[0]
#         tot_cnt += input_embeddings_batch.size()[0]
#         current_data = dataloader[phase].load_data()

#     epoch_loss = running_loss / tot_cnt

#     results[f"{phase}_loss"] = epoch_loss
# results["model"] = model

## Train Code

In [16]:
import EEG_TEXT_BART_SENTIMENT
from tqdm import tqdm

In [22]:
dataloader = dataset_dict["ZuCo-BART"]
model = model
# model.load_state_dict(torch.load("BUF.pt")
optimizer = optim.Adam(model.parameters(), lr=5e-4)
from transformers import BartTokenizer
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
criterion = nn.CrossEntropyLoss()
device = "cuda"
device_ids = None

In [25]:
all_results = []
for epoch in tqdm(range(50)):
    args_dict = {
        "dataloader" : dataset_dict["ZuCo-BART"],
        "model" : model,
        "optimizer" : optim.Adam(model.parameters(), lr=1e-3),
        "criterion" : nn.CrossEntropyLoss(),
        "tokenizer" : tokenizer,
        "device" : "cuda",
        "device_ids" : None,
    }
    results = EEG_TEXT_BART_SENTIMENT.train(args_dict)
    model = results["model"]
    print(epoch, results["train_loss"], results["dev_loss"])
    if epoch % 10 == 0:
        args_dict = {
            "dataloader" : dataset_dict["ZuCo-BART"],
            "model" : model,
            "optimizer" : optim.Adam(model.parameters(), lr=5e-4),
            "criterion" : nn.CrossEntropyLoss(),
            "tokenizer" : tokenizer,
            "device" : "cuda",
            "device_ids" : None
        }
        results = EEG_TEXT_BART_SENTIMENT.evaluate(args_dict)
        all_results.append(results)
        print()
        print(results)

  0%|          | 0/50 [00:00<?, ?it/s]

0 1.0975614288122786 1.096554949219071


/content/drive/MyDrive/Research/FINAL/Code/MMMM/./trainer/EEG_TEXT_BART_SENTIMENT.py:192: RuntimeWarning: invalid value encountered in divide
  results[f"{phase}_precision"] = precision = tp / (tp + fp)
  2%|▏         | 1/50 [00:25<20:40, 25.32s/it]


{'train_confusion_matrix': array([[   0,    0, 1111],
       [   0,    0, 1184],
       [   0,    0, 1250]]), 'train_TP': array([   0,    0, 1250]), 'train_FP': array([   0,    0, 2295]), 'train_TN': array([2434, 2361,    0]), 'train_FN': array([1111, 1184,    0]), 'train_accuracy': array([0.68660085, 0.66600846, 0.35260931]), 'train_precision': array([       nan,        nan, 0.35260931]), 'train_recall': array([0., 0., 1.]), 'train_f1': array([       nan,        nan, 0.52137643]), 'dev_confusion_matrix': array([[  0,   0, 108],
       [  0,   0, 219],
       [  0,   0, 139]]), 'dev_TP': array([  0,   0, 139]), 'dev_FP': array([  0,   0, 327]), 'dev_TN': array([358, 247,   0]), 'dev_FN': array([108, 219,   0]), 'dev_accuracy': array([0.76824034, 0.53004292, 0.29828326]), 'dev_precision': array([       nan,        nan, 0.29828326]), 'dev_recall': array([0., 0., 1.]), 'dev_f1': array([       nan,        nan, 0.45950413])}


  4%|▍         | 2/50 [00:40<15:26, 19.29s/it]

...

47 1.0975254488877528 1.0966362166325838


 98%|█████████▊| 49/50 [13:13<00:15, 15.40s/it]

48 1.0975250785499706 1.0966394398179384


100%|██████████| 50/50 [13:29<00:00, 16.18s/it]

49 1.0975248220499199 1.0966425369206496


In [26]:
all_results[-1]

{'train_confusion_matrix': array([[   0,    0, 1111],
        [   0,    0, 1184],
        [   0,    0, 1250]]),
 'train_TP': array([   0,    0, 1250]),
 'train_FP': array([   0,    0, 2295]),
 'train_TN': array([2434, 2361,    0]),
 'train_FN': array([1111, 1184,    0]),
 'train_accuracy': array([0.68660085, 0.66600846, 0.35260931]),
 'train_precision': array([       nan,        nan, 0.35260931]),
 'train_recall': array([0., 0., 1.]),
 'train_f1': array([       nan,        nan, 0.52137643]),
 'dev_confusion_matrix': array([[  0,   0, 108],
        [  0,   0, 219],
        [  0,   0, 139]]),
 'dev_TP': array([  0,   0, 139]),
 'dev_FP': array([  0,   0, 327]),
 'dev_TN': array([358, 247,   0]),
 'dev_FN': array([108, 219,   0]),
 'dev_accuracy': array([0.76824034, 0.53004292, 0.29828326]),
 'dev_precision': array([       nan,        nan, 0.29828326]),
 'dev_recall': array([0., 0., 1.]),
 'dev_f1': array([       nan,        nan, 0.45950413])}